In [ ]:
%load_ext autoreload
%autoreload 2

import os
import yaml
import json
import polars as pl
import pandas as pd
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

from anngeno import AnnGeno
from scripts import get_burdens
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(num_cores)

## Get missense variants for required gene

In [ ]:
pg = pl.read_parquet('/home/dnanexus/data_dir/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet').filter(pl.col('gene_name').is_in(['GCK']))
pg

In [ ]:
pg['file_name'].value_counts().sort('count', descending=True)

In [ ]:
pg = pg.with_columns(
    pl.when(pl.col('file_name') == 'HXK4_HUMAN_Gersing_2022_activity')
      .then(pl.lit('dms_activity'))
      .when(pl.col('file_name') == 'HXK4_HUMAN_Gersing_2023_abundance')
      .then(pl.lit('dms_abundance'))
      .otherwise(None)
      .alias('score_type')
)
pg

In [ ]:
(
    ggplot(pg, aes(x='dms_score', fill='score_type')) +
    geom_histogram(bins=100, alpha=0.5, position='identity') +
    theme_bw()
)

In [ ]:
# We have two types of DMS scores: activity and abundance.
pg = pg.pivot(
    index = ['mutant', 'region', 'gene_name'],
    on = 'score_type',
    values = 'dms_score'
).drop_nulls()
pg

In [ ]:
anno = pl.read_parquet('/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag/annotations.parquet').with_columns(
    pl.when(
        pl.col('amino_acids').is_not_null() & pl.col('protein_position').is_not_null()
    ).then(
        pl.col('amino_acids').str.split('/').list.get(0) +
        pl.col('protein_position').str.split('/').list.get(0) +
        pl.col('amino_acids').str.split('/').list.get(1)
    ).otherwise(None).alias('mutant')
).filter(
    # Filter for GCK gene
    pl.col('region').is_in(['ENSG00000106633'])
)

id_cols = ['chrom', 'pos', 'ref', 'alt', 'id', 'region', 'col', 'AF_ukb', 'mutant', 'amino_acids', 'protein_position', 'consequence']
missense_annos = ['loftee_hc', 'CADD_RAW', 'am_pathogenicity', 'Consequence_missense_variant', 'PolyPhen', 'CADD_SIFTval', 'CADD_priPhCons', 'CADD_mamPhCons', 'CADD_verPhCons', 'gpn_score']

anno = anno.select(id_cols + missense_annos)
anno

In [ ]:
pg.filter(~pl.col('mutant').is_in(anno['mutant']))

In [ ]:
brca_df = pg.join(anno.filter(pl.col('region') == 'ENSG00000106633'), on=['region', 'mutant'], how='inner')

annos2compare = missense_annos + ['dms_activity', 'dms_abundance']
brca_df

### Check how exp. scores correlate with comp. scores

In [ ]:
(
    ggplot(brca_df, aes(x='dms_activity', y='am_pathogenicity')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS activity', y='Alphamissense') +
    theme_bw()
)

In [ ]:
(
    ggplot(brca_df, aes(x='dms_abundance', y='am_pathogenicity')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS abundance', y='Alphamissense') +
    theme_bw()
)

### Pos-neg split scores

In [ ]:

def check_positive_negative(df: pl.DataFrame, columns):
    results = {}
    for col in columns:
        if col in df.columns:
            non_null = df.select(pl.col(col).drop_nulls())[col]
            if non_null.is_empty():
                results[col] = False  # Only nulls
            else:
                min_val = non_null.min()
                max_val = non_null.max()
                results[col] = (min_val < 0) and (max_val > 0)
        else:
            results[col] = False  # Column not found
    return results

# Example usage:
positive_negative_check = check_positive_negative(brca_df, annos2compare)

# Print the results
for column, has_both in positive_negative_check.items():
    if has_both:
        print(f"Column '{column}': Contains both positive and negative values.")

In [ ]:
def split_pos_neg_lazy(df: pl.LazyFrame, columns):
    # Start with the lazy frame
    lf = df

    for col in columns:
        if col in df.columns:
            pos_col = (
                pl.when(pl.col(col) > 0)
                .then(pl.col(col))
                .otherwise(0)
                .alias(f"{col}_pos")
            )

            neg_col = (
                pl.when(pl.col(col) < 0)
                .then(pl.col(col))
                .otherwise(0)
                .alias(f"{col}_neg")
            )

            lf = lf.with_columns([pos_col, neg_col])

    return lf

# Get the columns that contain both positive and negative values
mix_cols = [k for k, v in positive_negative_check.items() if v]

# Split the positive and negative values into separate columns
split_ann = split_pos_neg_lazy(brca_df.lazy(), mix_cols).collect()
split_ann

## Compute burdens

In [ ]:
split_ann

In [ ]:
%%time

config_path = "/home/dnanexus/ukbgym/config_dms.yaml"
output_dir = "/home/dnanexus/data_dir/dms_burdens/"
gene_id = 'ENSG00000106633'

get_burdens.compute_and_store_burdens(
    config_path=config_path,
    gene_list=[gene_id],
    output_dir=output_dir,
    only_snps=True,
    na_mask=True,
    overwrite=True,
    gene_chunk_size=1,
    sample_chunk_size=50_000,
    new_annotation_df=split_ann.lazy(),
    variant_subset=split_ann['id'].unique().to_list()
)

In [ ]:
b = pl.read_parquet(f'{output_dir}/{gene_id}.parquet')
b.drop_nans()

## get phenotypes

In [ ]:
phenos = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/phenotypes190_missing80_unique2.parquet').select(['individual', 'glycated_haemoglobin_hba1c']).drop_nulls()
phenos

In [ ]:
prs = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/PRS190_missing80_unique2.parquet').select(['individual', 'glycated_haemoglobin_hba1c_prs']).drop_nulls()
plt.hist(prs['glycated_haemoglobin_hba1c_prs'], bins=100)
plt.show()

In [ ]:
config_path = f'/home/dnanexus/ukbgym/config_dms.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

cov_list = config.get("covariates")

cov_df = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet').rename({'eid':'individual'}).select(['individual'] + cov_list).with_columns(pl.col('individual').cast(pl.Int64).alias('individual'))
cov_df

### correct for PRS and covariates

In [ ]:
all_df = phenos.join(prs, on='individual', how='inner').join(cov_df, on='individual', how='inner')
all_pd = all_df.to_pandas()
all_pd

In [ ]:
# Restrict to EUR ancestry
eur_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv')
eur_samples

all_pd = all_pd[all_pd['individual'].isin(eur_samples['eid'].to_list())]
all_pd

In [ ]:
import statsmodels.api as sm

pheno = 'glycated_haemoglobin_hba1c'

combined_df = pd.DataFrame(index=all_pd.index)

y = all_pd[pheno]
X = all_pd.drop(columns=[pheno])
X = sm.add_constant(X)  # Add a constant term for the intercept

# Fit the model
model = sm.OLS(y, X).fit()

# Save residuals
residuals = pd.Series(model.resid, index=combined_df.index, name=f'{pheno}_residual')

pdf = pl.DataFrame(pd.concat([all_pd[['individual', pheno, f'{pheno}_prs']], residuals], axis=1)).with_columns(pl.col('individual').cast(pl.String).alias('individual'))
pdf

## Merge and Plot

In [ ]:
gene_id = 'ENSG00000106633'
gis = pl.read_parquet(f'/home/dnanexus/data_dir/dms_burdens/{gene_id}.parquet').rename({'sample_id':'individual'})
gis

In [ ]:
plt_df = gis.join(pdf, on='individual', how='inner').to_pandas()
plt_df

In [ ]:
plt_df.dropna()

In [ ]:
from scipy.stats import pearsonr

anno = 'am_pathogenicity'

df_filtered = plt_df[plt_df['annotation'] == anno].dropna(subset=['max', 'glycated_haemoglobin_hba1c_residual'])
corr, pval = pearsonr(df_filtered['max'], df_filtered['glycated_haemoglobin_hba1c_residual'])
corr_text = f'corr. = {corr:.2f}, p = {pval:.3g}'

(
    ggplot(df_filtered, aes(x='max', y='glycated_haemoglobin_hba1c_residual')) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=True, color='darkred') +
    theme_bw() +
    labs(x='Alphamissense', y='glycated haemoglobin (HbA1c) residual') +
    annotate('text', x=df_filtered['max'].min(), y=df_filtered['glycated_haemoglobin_hba1c_residual'].max(), label=corr_text, ha='left', va='top', size=12)
)

In [ ]:
anno='dms_activity_pos'

df_filtered = plt_df[plt_df['annotation'] == anno].dropna(subset=['max', 'glycated_haemoglobin_hba1c_residual'])
corr, pval = pearsonr(df_filtered['max'], df_filtered['glycated_haemoglobin_hba1c_residual'])
corr_text = f'corr. = {corr:.2f}, p = {pval:.3g}'

(
    ggplot(df_filtered, aes(x='max', y='glycated_haemoglobin_hba1c_residual')) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=True, color='darkred') +
    theme_bw() +
    labs(x='DMS activity', y='glycated haemoglobin (HbA1c) residual') +
    annotate('text', x=df_filtered['max'].min(), y=df_filtered['glycated_haemoglobin_hba1c_residual'].max(), label=corr_text, ha='left', va='top', size=12)
)

In [ ]:
anno = 'dms_abundance_pos'

df_filtered = plt_df[plt_df['annotation'] == anno].dropna(subset=['max', 'glycated_haemoglobin_hba1c_residual'])
corr, pval = pearsonr(df_filtered['max'], df_filtered['glycated_haemoglobin_hba1c_residual'])
corr_text = f'corr. = {corr:.2f}, p = {pval:.3g}'

(
    ggplot(df_filtered, aes(x='max', y='glycated_haemoglobin_hba1c_residual')) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=True, color='darkred') +
    theme_bw() +
    labs(x='DMS abundance', y='glycated haemoglobin (HbA1c) residual') +
    annotate('text', x=df_filtered['max'].min(), y=df_filtered['glycated_haemoglobin_hba1c_residual'].max(), label=corr_text, ha='left', va='top', size=12)
)

In [ ]:
import numpy as np
corr_results = []

# Group by annotation and calculate correlation
for annotation, group in plt_df.groupby('annotation'):
    # Drop NA in the relevant columns
    clean_group = group.dropna(subset=['max', 'glycated_haemoglobin_hba1c_residual'])
    if len(clean_group) > 1:  # Need at least 2 points to calculate correlation
        corr, pval = pearsonr(clean_group['max'], clean_group['glycated_haemoglobin_hba1c_residual'])
        corr_results.append({'annotation': annotation, 'abs_correlation': np.abs(corr), 'p_value': pval})
    else:
        corr_results.append({'annotation': annotation, 'abs_correlation': None, 'p_value': None})

# Convert to DataFrame
corr_df = pd.DataFrame(corr_results).dropna()

drop_anno = ['dms_activity_neg','dms_abundance_neg','CADD_RAW_neg','gpn_score_pos']
corr_df = corr_df[~corr_df['annotation'].isin(drop_anno)].sort_values('abs_correlation', ascending=False)
corr_df['annotation'] = pd.Categorical(corr_df['annotation'], categories=corr_df['annotation'].unique(), ordered=True)

corr_df['color_dms'] = corr_df['annotation'].str.startswith('dms_').map({True: 'DMS', False: 'Other'})

# Plot bar plot of correlations
(
    ggplot(corr_df, aes(x='annotation', y='abs_correlation', fill='color_dms')) +
    geom_col(alpha=0.8) +
    coord_flip() +  # Flip for better readability if many annotations
    theme_bw() +
    labs(
        x='Annotation',
        y='absolute Pearson correlation'
    ) +
    theme(
        figure_size=(8, 3),
        legend_position='none'
    )
)
